# Filter

In this notebook, we present different ways of filtering a large dataset: keeping only the rows that match a condition.

**Learning outcomes**

- Build a boolean mask from a condition and use it with `.loc[mask, columns]`.
- Combine conditions with `&` (and), `|` (or) and `~` (not), using parentheses.
- Use `isin`, `between` and `str.contains` for common filters.
- Recognise missing values (`NaN`) and filter on them with `isna()` / `notna()`.
- Read the output of `value_counts()` and compute a simple ratio.

**Before you start:** you should be comfortable with `.loc`, `[]` and the idea of a `True`/`False` mask per row (notebook `09_Select`).

These notebooks are tested with **pandas 3.x**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from course_checks import check, check_equal

print(pd.__version__)

**You should see:** a version starting with `3.`.

Let's continue with our NYC 311 service requests example. (We declare `Incident Zip` as text, as we learned in the previous notebook, to avoid the `DtypeWarning`.)

In [ ]:
complaints = pd.read_csv('311-service-requests.csv', dtype={'Incident Zip': str})
complaints.shape

**You should see:** `(111069, 52)`.

In [ ]:
complaints.head()

In [ ]:
complaints.dtypes.head(10)

**You should see:** `str` for text columns (pandas 3; older versions print `object`) and `int64` for `Unique Key`.

## Gathering only a selected category

Suppose we want to investigate the samples whose Complaint Type is 'Noise - Street/Sidewalk'.

Before we continue, I suggest we first check the unique values in the column.

In [ ]:
cats = complaints['Complaint Type'].unique()

print("Total: ", len(cats))
cats[:10]

**You should see:** `Total:  165` followed by the first 10 complaint types, such as `'Noise - Street/Sidewalk'`, `'Illegal Parking'`, ...

We can see that there are a lot of categories! This is confusing.

For now, let's focus on simplicity. Let's say we want to see 'Noise - Street/Sidewalk'. Pass a condition inside `[]`:

In [ ]:
complaints[complaints['Complaint Type'] == 'Noise - Street/Sidewalk']

**You should see:** a table that ends with `1928 rows x 52 columns`, where every row has `Noise - Street/Sidewalk` as complaint type.

If you look at the result, you'll see that this worked, and it only contains complaints with the right complaint type. But how does this work? Let's deconstruct it into two pieces.

In [ ]:
complaints['Complaint Type'] == "Noise - Street/Sidewalk"

**You should see:** a Series of `True`/`False`, 111069 of them, named `Complaint Type`, with `dtype: bool`.

This is a big array of Trues and Falses, one for each row in our DataFrame. When we index our DataFrame with this array, we get just the rows where our boolean array evaluated to `True`. The mask must have the same length (and index) as the DataFrame. We call such a Series of booleans a **mask**.

We will write filters in the form **`df.loc[mask, columns]`**: the mask picks the rows and the second part the columns, in a single step.

## Combining conditions

You can combine more than one condition with the `&` (and) operator:

In [ ]:
is_noise = complaints['Complaint Type'] == "Noise - Street/Sidewalk"
in_brooklyn = complaints['Borough'] == "BROOKLYN"
complaints[is_noise & in_brooklyn]

**You should see:** `456 rows x 52 columns`.

Or if we just wanted a few columns, put them after the mask in `.loc`. The older style `complaints[mask][[columns]]` also works, but it is a chained selection (see the previous notebook): it builds the full 52-column table first, and it is not safe if you later want to assign values.

In [ ]:
complaints.loc[is_noise & in_brooklyn, ['Complaint Type', 'Borough', 'Created Date', 'Descriptor']]

### Operators: `&`, `|`, `~` and parentheses

Python's words `and`, `or`, `not` do not work on masks. Use these symbols instead:

| Meaning | Operator | Example |
|---|---|---|
| and | `&` | `(a == 1) & (b == 2)` |
| or | `\|` | `(a == 1) \| (b == 2)` |
| not | `~` | `~(a == 1)` |

**Always put each condition in parentheses** when you write them inline, because `&` and `|` bind more tightly than `==`: `a == 1 & b == 2` is evaluated in the wrong order and fails with a confusing error.

In [ ]:
bronx_or_brooklyn = complaints.loc[(complaints['Borough'] == 'BRONX') | (complaints['Borough'] == 'BROOKLYN')]
not_noise = complaints.loc[~is_noise]

print(len(bronx_or_brooklyn), len(not_noise), len(complaints) - is_noise.sum())

**You should see:** `52576 109141 109141`. The last two numbers agree: `~is_noise` keeps exactly the rows that are *not* noise.

### `isin`: one of several values

The `|` version above gets long with many values. `isin` takes a list and does the same job:

In [ ]:
bronx_or_brooklyn2 = complaints.loc[complaints['Borough'].isin(['BRONX', 'BROOKLYN'])]
print(len(bronx_or_brooklyn2))
bronx_or_brooklyn2.equals(bronx_or_brooklyn)

**You should see:** `52576` and `True`: both ways give identical tables.

### `str.contains`: text that contains a word

For *partial* text matches use the `.str` accessor. How many different kinds of noise complaint exist? They all have "Noise" in the name, but with different endings.

Two details: `case=False` ignores capital letters, and `na=False` treats missing text as "does not match" (otherwise the mask would contain `NaN`/missing values and filtering with it would fail).

In [ ]:
has_noise = complaints['Complaint Type'].str.contains('noise', case=False, na=False)
complaints.loc[has_noise, 'Complaint Type'].value_counts()

**You should see:** 8 complaint types. The largest are `Noise` (3321), `Noise - Commercial` (2578) and `Noise - Street/Sidewalk` (1928); the smallest is `Collection Truck Noise` (8). The sum of all counts is 8942.

### `between`: a numeric range

`between(low, high)` is a shorter way to write `(s >= low) & (s <= high)`; both ends are inclusive.

In [ ]:
key = complaints['Unique Key']
in_range = key.between(26500000, 26550000)
same_thing = (key >= 26500000) & (key <= 26550000)
print(in_range.sum(), (in_range == same_thing).all())

**You should see:** `32342 True`: 32,342 complaints in that range, and `True`: both ways agree on every row.

## Missing values

Not every cell has a value. Missing data is shown as `NaN` ("Not a Number"; for text columns pandas 3 also uses `NaN`). For example, a complaint that is still open has no `Closed Date`.

In [ ]:
complaints[['Unique Key', 'Agency', 'Status', 'Closed Date']].head(8)

**You should see:** a few rows with `NaN` in `Closed Date` and the `Status` for those rows is not `Closed` (for example `Assigned` or `Open`).

**You must not test for missing values with `== np.nan`** (it is always `False`). Use `isna()` (is it missing?) and `notna()` (is it present?). They return masks, so you can count them (`True` counts as 1) or filter with them.

In [ ]:
no_closed = complaints['Closed Date'].isna()
print('missing Closed Date:', no_closed.sum())
print('present Closed Date:', complaints['Closed Date'].notna().sum())
complaints.loc[no_closed, ['Agency', 'Status', 'Closed Date']].head()

**You should see:**

```text
missing Closed Date: 50799
present Closed Date: 60270
```

and 5 rows of open complaints whose `Closed Date` is `NaN`. The two numbers add up to 111069.

What to *do* with missing values (convert dates properly, drop or fill them) is the subject of the next notebook, `11_Dates_and_Missing_Values`.

## Which borough has the most noise complaints?

In [ ]:
is_noise = complaints['Complaint Type'] == "Noise - Street/Sidewalk"
noise_complaints = complaints.loc[is_noise]
noise_complaints['Borough'].value_counts()

**You should see:**

```text
Borough
MANHATTAN        917
BROOKLYN         456
BRONX            292
QUEENS           226
STATEN ISLAND     36
Unspecified        1
Name: count, dtype: int64
```

Notice the shape of this result in pandas 3: it is a **Series** whose *values* are the counts and whose *name* is `count`; the *index* holds the borough names and carries the name of the original column, `Borough`. (In pandas 1.x the name was the column's name and the index had no name, so older tutorials look different.) `value_counts()` sorts from most to least frequent.

It's Manhattan! But what if we wanted to divide by the total number of complaints, to make it make a bit more sense? That would be easy too. Both Series are indexed by borough, so pandas matches the boroughs up for us before dividing. (No need for `astype(float)`: `/` already gives decimals.)

In [ ]:
noise_complaint_counts = noise_complaints['Borough'].value_counts()
complaint_counts = complaints['Borough'].value_counts()
noise_complaint_counts / complaint_counts

**You should see:** fractions between 0 and 1: about `0.038` for `MANHATTAN`, about `0.015` for `BRONX` and `0.014` for `BROOKLYN`, `0.010` for `QUEENS`, `0.007` for `STATEN ISLAND` and almost `0.000` for `Unspecified`. So about 3.8% of Manhattan's complaints are about street noise, versus about 1% in the other boroughs.

Let's plot it.

In [ ]:
ratio = noise_complaint_counts / complaint_counts
ax = ratio.sort_values().plot(kind='bar', title='Share of complaints that are street/sidewalk noise')
ax.set_xlabel('Borough')
ax.set_ylabel('Fraction of all complaints')
plt.show()

**You should see:** a bar chart with 6 bars, sorted from the smallest (`Unspecified`, almost invisible) to the tallest (`MANHATTAN`, around 0.038), with a title and axis labels.

There is an easier way to compute this kind of "per group" summary than filtering each category one by one: **`groupby`**, which you will meet in notebook `12_GroupBy`.

### ✏️ Your turn

What are the **top 5 complaint types in the BRONX**? Filter the `Borough` with `.loc`, take the `Complaint Type` column, and use `value_counts().head(5)`. Store the result in `top5`.

In [ ]:
top5 = ...  # <- replace ...

In [ ]:
check_equal(lambda: top5.index.tolist(), ['HEATING', 'GENERAL CONSTRUCTION', 'PAINT - PLASTER', 'Street Light Condition', 'PLUMBING'], hint="complaints.loc[complaints['Borough'] == 'BRONX', 'Complaint Type'].value_counts().head(5)")
check_equal(lambda: top5.iloc[0], 4093, hint='HEATING should come first with 4093 complaints')

<details>
<summary>Show solution</summary>

```python
top5 = complaints.loc[complaints['Borough'] == 'BRONX', 'Complaint Type'].value_counts().head(5)
```

</details>

### ✏️ Your turn

How many complaints handled by the agency `NYPD` have **no** `Closed Date`? Build a mask with `==` and `isna()`, combine them with `&`, and store the count in `nypd_open`.

In [ ]:
nypd_open = ...  # <- replace ...

In [ ]:
check_equal(lambda: int(nypd_open), 47, hint="((complaints['Agency'] == 'NYPD') & complaints['Closed Date'].isna()).sum()")

<details>
<summary>Show solution</summary>

```python
nypd_open = ((complaints['Agency'] == 'NYPD') & complaints['Closed Date'].isna()).sum()
```

</details>

### ✏️ Your turn

Keep the rows whose `Agency` is in the list `['HPD', 'DOT']` **and** whose `Borough` is not `'Unspecified'`. Use `isin`, `~` and `.loc`, keeping only the columns `Agency`, `Borough` and `Complaint Type`. Store the result in `selected`.

In [ ]:
selected = ...  # <- replace ...

In [ ]:
check_equal(lambda: selected.shape, (56454, 3), hint="Combine (isin) & (~ ... == 'Unspecified') with parentheses")
check_equal(lambda: selected.columns.tolist(), ['Agency', 'Borough', 'Complaint Type'], hint='the second part of .loc is the column list')

<details>
<summary>Show solution</summary>

```python
selected = complaints.loc[complaints['Agency'].isin(['HPD', 'DOT']) & ~(complaints['Borough'] == 'Unspecified'), ['Agency', 'Borough', 'Complaint Type']]
```

</details>

### ✏️ Your turn

How many complaints have a `Complaint Type` containing the word `'Noise'` (any capitalisation) **and** `Borough` equal to `'MANHATTAN'`? Use `str.contains(..., case=False, na=False)` and store the count in `manhattan_noise`.

In [ ]:
manhattan_noise = ...  # <- replace ...

In [ ]:
check_equal(lambda: int(manhattan_noise), 4337, hint="(contains mask & (Borough == 'MANHATTAN')).sum()")

<details>
<summary>Show solution</summary>

```python
manhattan_noise = (complaints['Complaint Type'].str.contains('noise', case=False, na=False) & (complaints['Borough'] == 'MANHATTAN')).sum()
```

</details>

## ✅ Check yourself

- I can write a filter as `df.loc[mask, columns]` and explain why it is better than `df[mask][columns]`.
- I can combine conditions with `&`, `|` and `~`, and I know each condition needs parentheses.
- I can choose between `==`, `isin`, `between` and `str.contains(..., na=False)`.
- I know missing values are `NaN`, and I test for them with `isna()` / `notna()`, never with `==`.
- I can read `value_counts()` output: a Series called `count`, indexed by the category.

## Next

Our dates are still plain text and many cells are missing. In `11_Dates_and_Missing_Values` you will convert text to real dates and decide what to do with `NaN`. After that, `12_GroupBy` shows the easier way to do the borough ratio.